In [2]:
# Import libraries
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.dates import DateFormatter
import numpy as np

In [3]:
df = pd.read_csv('rideshare_kaggle.csv')
np.random.seed(42)

In [4]:
df.head()

,id,hour,day,month,source,destination,cab_type,product_id,name,price,...,pressure,windBearing,cloudCover,uvIndex,ozone,moonPhase,temperatureMin,temperatureMax,apparentTemperatureMin,apparentTemperatureMax
0,424553bb-7174-41ea-aeb4-fe06d4f4b9d7,9,16,12,Haymarket Square,North Station,Lyft,lyft_line,Shared,5.0,...,1021.98,57,0.72,0,303.8,0.30,39.89,43.68,33.73,38.07
1,4bd23055-6827-41c6-b23b-3c491f24e74d,2,27,11,Haymarket Square,North Station,Lyft,lyft_premier,Lux,11.0,...,1003.97,90,1.00,0,291.1,0.64,40.49,47.30,36.20,43.92
2,981a3613-77af-4620-a42a-0c0866077d1e,1,28,11,Haymarket Square,North Station,Lyft,lyft,Lyft,7.0,...,992.28,240,0.03,0,315.7,0.68,35.36,47.55,31.04,44.12
3,c2d88af2-d278-4bfd-a8d0-29ca77cc5512,4,30,11,Haymarket Square,North Station,Lyft,lyft_luxsuv,Lux Black XL,26.0,...,1013.73,310,0.00,0,291.1,0.75,34.67,45.03,30.30,38.53
4,e0126e1f-8ca9-4f2e-82b3-50505a09db9a,3,29,11,Haymarket Square,North Station,Lyft,lyft_plus,Lyft XL,9.0,...,998.36,303,0.44,0,347.7,0.72,33.10,42.18,29.11,35.75


In [5]:

df = pd.read_csv("rideshare_kaggle.csv")

df_od = df[["source", "destination", "day", "month"]].copy()

df_od["datetime"] = pd.to_datetime(
    dict(year=1900, month=df_od["month"], day=df_od["day"])
)
df_od["dayofweek"] = df_od["datetime"].dt.day_name()  # Monday, Tuesday, ...


## Part I

### 1-1

In [6]:

od_all = df_od.groupby(["source", "destination"]).size().unstack(fill_value=0)

print("=== OD matrix (all days) ===")
print(od_all)

od_all.to_csv("OD_matrix_all_days.csv")


=== OD matrix (all days) ===
destination              Back Bay  Beacon Hill  Boston University  Fenway  \
source                                                                      
Back Bay                        0            0               9435    9469   
Beacon Hill                     0            0               9222    9436   
Boston University            9435         9215                  0       0   
Fenway                       9476         9436                  0       0   
Financial District              0            0               9711    9695   
Haymarket Square             9605         9566                  0       0   
North End                   10225        10039                  0       0   
North Station                   0            0               9480    9751   
Northeastern University      9579         9690                  0       0   
South Station                9460         9457                  0       0   
Theatre District                0            0 

### 1-2

In [7]:

od_by_dow = {}

for dow, sub in df_od.groupby("dayofweek"):
    od_dow = sub.groupby(["source", "destination"]).size().unstack(fill_value=0)
    od_by_dow[dow] = od_dow

    print(f"\n=== OD matrix for {dow} ===")
    print(od_dow)

    od_dow.to_csv(f"OD_matrix_{dow}.csv")



=== OD matrix for Friday ===
destination              Back Bay  Beacon Hill  Boston University  Fenway  \
source                                                                      
Back Bay                        0            0               1266    1200   
Beacon Hill                     0            0               1249    1309   
Boston University            1154         1165                  0       0   
Fenway                       1202         1373                  0       0   
Financial District              0            0               1247    1292   
Haymarket Square             1323         1387                  0       0   
North End                    1257         1213                  0       0   
North Station                   0            0               1062    1218   
Northeastern University      1236         1185                  0       0   
South Station                1229         1207                  0       0   
Theatre District                0            0

### 1-3

In [8]:

# مجموع خروجی از هر مبدأ
outgoing = df_od.groupby("source").size()
# مجموع ورودی به هر مقصد
incoming = df_od.groupby("destination").size()

zones = sorted(set(outgoing.index).union(set(incoming.index)))

balance_df = pd.DataFrame(index=zones)
balance_df["outgoing"] = outgoing.reindex(zones, fill_value=0)
balance_df["incoming"] = incoming.reindex(zones, fill_value=0)
balance_df["balance"] = balance_df["outgoing"] - balance_df["incoming"]

print("\n=== Flow balance for each zone (outgoing - incoming) ===")
print(balance_df)

# نواحی نامتعادل
unbalanced = balance_df[balance_df["balance"] != 0]
print("\n=== Zones without balance (outgoing != incoming) ===")
print(unbalanced)


=== Flow balance for each zone (outgoing - incoming) ===
                         outgoing  incoming  balance
Back Bay                    57792     57780       12
Beacon Hill                 57403     57403        0
Boston University           57764     57764        0
Fenway                      57757     57757        0
Financial District          58857     58851        6
Haymarket Square            57736     57764      -28
North End                   57763     57756        7
North Station               57118     57119       -1
Northeastern University     57756     57755        1
South Station               57750     57749        1
Theatre District            57813     57798       15
West End                    57562     57575      -13

=== Zones without balance (outgoing != incoming) ===
                         outgoing  incoming  balance
Back Bay                    57792     57780       12
Financial District          58857     58851        6
Haymarket Square            57736     57

## Part II

In [9]:
import numpy as np
import pandas as pd

# -----------------------------
# داده‌های ورودی برای بخش دوم
# -----------------------------
zones = [
    "Financial District",
    "Theatre District",
    "Back Bay",
    "Haymarket Square",
    "Boston University",
    "Fenway",
    "North End",
    "Northeastern University",
    "South Station",
    "West End",
    "Beacon Hill",
    "North Station",
]

od_base = od_all.reindex(index=zones, columns=zones).fillna(0).astype(float)

future_origins = pd.Series(
    [56850, 57820, 57799, 57836, 57714, 57750, 57963, 55756, 58750, 57560, 57483, 56228],
    index=zones,
    name="O_future"
)
future_destinations = pd.Series(
    [59700, 57900, 58600, 56840, 59500, 56750, 58750, 56950, 58700, 56600, 58200, 56200],
    index=zones,
    name="D_future"
)

tol = 0.01
max_iter = 1000

def relative_diff(target, current):
    target = target.astype(float)
    current = current.astype(float)
    mask = target != 0
    return np.max(np.abs(target[mask] - current[mask]) / target[mask])


### 2-1

In [10]:
# -----------------------------
# 2-1 Furness
# -----------------------------
def furness(od0, O_target, D_target, tol=0.01, max_iter=1000):
    T = od0.values.astype(float)
    O_target = O_target.values.astype(float)
    D_target = D_target.values.astype(float)
    history = []

    for _ in range(max_iter):
        row_sums = T.sum(axis=1)
        with np.errstate(divide="ignore", invalid="ignore"):
            factors_row = np.where(row_sums != 0, O_target / row_sums, 1.0)
        T = (T.T * factors_row).T

        col_sums = T.sum(axis=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            factors_col = np.where(col_sums != 0, D_target / col_sums, 1.0)
        T = T * factors_col

        row_diff = relative_diff(O_target, T.sum(axis=1))
        col_diff = relative_diff(D_target, T.sum(axis=0))
        err = max(row_diff, col_diff)
        history.append(err)
        if err < tol:
            break

    T_df = pd.DataFrame(T, index=od0.index, columns=od0.columns)
    return T_df, history

T_furness, hist_furness = furness(od_base, future_origins, future_destinations, tol, max_iter)
print("\n=== Furness OD matrix ===")
print(T_furness)



=== Furness OD matrix ===
destination              Financial District  Theatre District      Back Bay  \
source                                                                        
Financial District                 0.000000          0.000000      0.000000   
Theatre District                   0.000000          0.000000      0.000000   
Back Bay                           0.000000          0.000000      0.000000   
Haymarket Square               10316.254505       9694.031633   9777.741706   
Boston University               9864.084336       9999.965064   9583.734428   
Fenway                          9841.931343       9286.437366   9633.524507   
North End                       9522.348314       9592.899048  10425.743791   
Northeastern University         9453.374366       9347.137222   9399.406490   
South Station                  10702.007136       9979.529667   9779.849078   
West End                           0.000000          0.000000      0.000000   
Beacon Hill              

### 2-2

In [11]:

# -----------------------------
# 2-2 Detroit
# -----------------------------
def detroit(od0, O_target, D_target, tol_factor=0.01, max_iter=1000):
    T = od0.values.astype(float)
    O0 = T.sum(axis=1)
    D0 = T.sum(axis=0)
    gi = np.where(O0 != 0, O_target.values / O0, 1.0)
    hj = np.where(D0 != 0, D_target.values / D0, 1.0)
    history = []

    for _ in range(max_iter):
        T = (T.T * gi).T * hj
        O_curr = T.sum(axis=1)
        D_curr = T.sum(axis=0)
        gi_new = np.where(O_curr != 0, O_target.values / O_curr, 1.0)
        hj_new = np.where(D_curr != 0, D_target.values / D_curr, 1.0)
        max_dev = max(np.max(np.abs(gi_new - 1.0)),
                      np.max(np.abs(hj_new - 1.0)))
        history.append(max_dev)
        gi, hj = gi_new, hj_new
        if np.all((gi >= 0.99) & (gi <= 1.01)) and np.all((hj >= 0.99) & (hj <= 1.01)):
            break

    T_df = pd.DataFrame(T, index=od0.index, columns=od0.columns)
    return T_df, history, gi, hj

T_detroit, hist_detroit, gi_det, hj_det = detroit(
    od_base, future_origins, future_destinations, tol_factor=0.01, max_iter=max_iter
)
print("\n=== Detroit OD matrix ===")
print(T_detroit)



=== Detroit OD matrix ===
destination              Financial District  Theatre District      Back Bay  \
source                                                                        
Financial District                 0.000000          0.000000      0.000000   
Theatre District                   0.000000          0.000000      0.000000   
Back Bay                           0.000000          0.000000      0.000000   
Haymarket Square               10298.001270       9674.742007   9758.184035   
Boston University               9842.566421       9975.946861   9560.616520   
Fenway                          9819.470413       9263.197753   9609.316370   
North End                       9506.528929       9574.847195  10406.016486   
Northeastern University         9434.559506       9326.473412   9378.529600   
South Station                  10683.182071       9959.775155   9760.388337   
West End                           0.000000          0.000000      0.000000   
Beacon Hill              

### 2-3

In [12]:
# -----------------------------
# 2-3 Fratar
# -----------------------------
def fratar(od0, O_target, D_target, tol=0.01, max_iter=1000):
    T = od0.values.astype(float)
    O_target = O_target.values.astype(float)
    D_target = D_target.values.astype(float)
    history = []

    for _ in range(max_iter):
        O_curr = T.sum(axis=1)
        D_curr = T.sum(axis=0)
        gi = np.where(O_curr != 0, O_target / O_curr, 1.0)
        hj = np.where(D_curr != 0, D_target / D_curr, 1.0)
        T_new = (T.T * gi).T * hj
        row_diff = relative_diff(O_target, T_new.sum(axis=1))
        col_diff = relative_diff(D_target, T_new.sum(axis=0))
        err = max(row_diff, col_diff)
        history.append(err)
        T = T_new
        if err < tol:
            break

    T_df = pd.DataFrame(T, index=od0.index, columns=od0.columns)
    return T_df, history

T_fratar, hist_fratar = fratar(od_base, future_origins, future_destinations, tol, max_iter)
print("\n=== Fratar OD matrix ===")
print(T_fratar)



=== Fratar OD matrix ===
destination              Financial District  Theatre District      Back Bay  \
source                                                                        
Financial District                 0.000000          0.000000      0.000000   
Theatre District                   0.000000          0.000000      0.000000   
Back Bay                           0.000000          0.000000      0.000000   
Haymarket Square               10298.001270       9674.742007   9758.184035   
Boston University               9842.566421       9975.946861   9560.616520   
Fenway                          9819.470413       9263.197753   9609.316370   
North End                       9506.528929       9574.847195  10406.016486   
Northeastern University         9434.559506       9326.473412   9378.529600   
South Station                  10683.182071       9959.775155   9760.388337   
West End                           0.000000          0.000000      0.000000   
Beacon Hill               

### 2-4

In [13]:

# -----------------------------
# 2-4 Gravity models
# -----------------------------
# ماتریس فاصله میانگین از داده خام
dist_matrix = df.groupby(["source", "destination"])["distance"].mean().unstack()
dist_matrix = dist_matrix.reindex(index=zones, columns=zones).fillna(1.0)

def gravity_model(O_target, D_target, dist_matrix, beta=0.2, alpha=0.05,
                  tol=0.01, max_iter=1000):
    O = O_target.values.astype(float)
    D = D_target.values.astype(float)
    c = dist_matrix.values.astype(float)
    f = np.exp(-beta * c)
    T = np.outer(O, D / D.sum())
    history = []

    for _ in range(max_iter):
        row_factor = O / (T.sum(axis=1) + 1e-9)
        T = (T.T * row_factor).T
        T = T * f ** alpha
        col_factor = D / (T.sum(axis=0) + 1e-9)
        T = T * col_factor
        row_diff = relative_diff(O, T.sum(axis=1))
        col_diff = relative_diff(D, T.sum(axis=0))
        err = max(row_diff, col_diff)
        history.append(err)
        if err < tol:
            break

    T_df = pd.DataFrame(T, index=dist_matrix.index, columns=dist_matrix.columns)
    return T_df, history

T_grav_exp, hist_grav_exp = gravity_model(
    future_origins, future_destinations, dist_matrix,
    beta=0.2, alpha=0.05, tol=tol, max_iter=max_iter
)
print("\n=== Gravity OD matrix (exponential impedance) ===")
print(T_grav_exp)

def gravity_model_power(O_target, D_target, dist_matrix, gamma=1.5, alpha=0.05,
                        tol=0.01, max_iter=1000):
    O = O_target.values.astype(float)
    D = D_target.values.astype(float)
    c = dist_matrix.values.astype(float)
    f = 1.0 / (c ** gamma + 1e-9)
    T = np.outer(O, D / D.sum())
    history = []

    for _ in range(max_iter):
        row_factor = O / (T.sum(axis=1) + 1e-9)
        T = (T.T * row_factor).T
        T = T * f ** alpha
        col_factor = D / (T.sum(axis=0) + 1e-9)
        T = T * col_factor
        row_diff = relative_diff(O, T.sum(axis=1))
        col_diff = relative_diff(D, T.sum(axis=0))
        err = max(row_diff, col_diff)
        history.append(err)
        if err < tol:
            break

    T_df = pd.DataFrame(T, index=dist_matrix.index, columns=dist_matrix.columns)
    return T_df, history

T_grav_pow, hist_grav_pow = gravity_model_power(
    future_origins, future_destinations, dist_matrix,
    gamma=1.5, alpha=0.05, tol=tol, max_iter=max_iter
)
print("\n=== Gravity OD matrix (power impedance) ===")
print(T_grav_pow)



=== Gravity OD matrix (exponential impedance) ===
destination              Financial District  Theatre District     Back Bay  \
source                                                                       
Financial District              6438.425142       6160.009598  6293.173030   
Theatre District                6391.668337       6115.274689  6247.471067   
Back Bay                        6249.825089       5979.565140  6108.827830   
Haymarket Square                5176.037472       4200.331688  2217.992287   
Boston University                691.589511       2030.661982  5130.878426   
Fenway                           795.539002       2369.633451  5210.055456   
North End                       5437.445165       4025.494153  1661.062754   
Northeastern University          820.929032       3271.295848  4974.650871   
South Station                   8169.503756       5063.186258  1667.429634   
West End                        6495.570447       6214.683780  6349.029126   
Beacon Hill  

### 2-5

In [14]:

# -----------------------------
# 2-5 مقایسه همگرایی
# -----------------------------
print("\n=== Convergence summary (iterations) ===")
print(f"Furness:         {len(hist_furness)} iters, final error = {hist_furness[-1]:.5f}")
print(f"Detroit:         {len(hist_detroit)} iters, final max growth dev = {hist_detroit[-1]:.5f}")
print(f"Fratar:          {len(hist_fratar)} iters, final error = {hist_fratar[-1]:.5f}")
print(f"Gravity (exp):   {len(hist_grav_exp)} iters, final error = {hist_grav_exp[-1]:.5f}")
print(f"Gravity (power): {len(hist_grav_pow)} iters, final error = {hist_grav_pow[-1]:.5f}")


=== Convergence summary (iterations) ===
Furness:         1000 iters, final error = 0.01091
Detroit:         1 iters, final max growth dev = 0.00839
Fratar:          1 iters, final error = 0.00832
Gravity (exp):   68 iters, final error = 0.00999
Gravity (power): 1000 iters, final error = 0.03147


## Part III

In [15]:
df = pd.read_csv("rideshare_kaggle.csv")

df_od = df[["source", "destination", "day", "month", "cab_type", "price", "distance"]].copy()

df_od["datetime"] = pd.to_datetime(
    dict(year=1900, month=df_od["month"], day=df_od["day"])
)
df_od["dayofweek"] = df_od["datetime"].dt.day_name()  # Monday, Tuesday, ...

In [16]:
# تعریف بازه‌های زمانی ۶ ساعته
bins = [0, 6, 12, 18, 24]
labels = ['Night(0-6)', 'Morning(6-12)', 'Afternoon(12-18)', 'Evening(18-24)']

# ایجاد ستون hour (فرض می‌کنیم ستون ساعت دارید. اگر ندارید، باید ستون hour اضافه کنید)
# اگر ستون hour ندارید، می‌توانید از زمان تصادفی استفاده کنید یا ستونی دیگر
# برای این مثال فرض می‌کنیم ستون hour وجود دارد
df_od['hour'] = df_od['datetime'].dt.hour  # اگر ستون ساعت ندارید باید آن را ایجاد کنید

# ایجاد بازه زمانی
df_od['time_period'] = pd.cut(df_od['hour'], bins=bins, labels=labels, right=False, include_lowest=True)

In [17]:
# گروه‌بندی بر اساس مبدا، مقصد، نوع تاکسی، روز هفته و بازه زمانی
grouped = df_od.groupby(['source', 'destination', 'cab_type', 'dayofweek', 'time_period']).agg({
    'price': 'mean',
    'distance': 'mean'
}).reset_index()

# تغییر نام ستون‌ها
grouped = grouped.rename(columns={
    'price': 'avg_price',
    'distance': 'avg_distance'
})

In [18]:
grouped

,source,destination,cab_type,dayofweek,time_period,avg_price,avg_distance
0,Back Bay,Boston University,Lyft,Friday,Night(0-6),15.718805,1.495940
1,Back Bay,Boston University,Lyft,Monday,Night(0-6),14.663750,1.483712
2,Back Bay,Boston University,Lyft,Saturday,Night(0-6),15.250511,1.485337
3,Back Bay,Boston University,Lyft,Sunday,Night(0-6),14.312160,1.488348
4,Back Bay,Boston University,Lyft,Thursday,Night(0-6),14.955140,1.486112
...,...,...,...,...,...,...,...
1003,West End,South Station,Uber,Saturday,Night(0-6),14.617021,2.002819
1004,West End,South Station,Uber,Sunday,Night(0-6),14.479010,2.010715
1005,West End,South Station,Uber,Thursday,Night(0-6),14.154671,2.008314
1006,West End,South Station,Uber,Tuesday,Night(0-6),14.304613,1.998894


In [19]:
# جدول‌های جداگانه برای Lyft و Uber
lyft_df = grouped[grouped['cab_type'] == 'Lyft'].copy()
uber_df = grouped[grouped['cab_type'] == 'Uber'].copy()

# تغییر نام ستون‌ها برای ادغام
lyft_df = lyft_df.rename(columns={
    'avg_price': 'avg_price_lyft',
    'avg_distance': 'avg_distance_lyft'
})

uber_df = uber_df.rename(columns={
    'avg_price': 'avg_price_uber',
    'avg_distance': 'avg_distance_uber'
})

# ادغام دو دیتافریم
comparison_df = pd.merge(
    lyft_df[['source', 'destination', 'dayofweek', 'time_period', 'avg_price_lyft', 'avg_distance_lyft']],
    uber_df[['source', 'destination', 'dayofweek', 'time_period', 'avg_price_uber', 'avg_distance_uber']],
    on=['source', 'destination', 'dayofweek', 'time_period'],
    how='outer'
)

comparison_df = comparison_df[~comparison_df['dayofweek'].isin(['Saturday', 'Sunday'])]

In [20]:
comparison_df

,source,destination,dayofweek,time_period,avg_price_lyft,avg_distance_lyft,avg_price_uber,avg_distance_uber
0,Back Bay,Boston University,Friday,Night(0-6),15.718805,1.495940,12.781302,1.380631
1,Back Bay,Boston University,Monday,Night(0-6),14.663750,1.483712,13.361794,1.391154
4,Back Bay,Boston University,Thursday,Night(0-6),14.955140,1.486112,13.084733,1.402770
5,Back Bay,Boston University,Tuesday,Night(0-6),15.240453,1.487482,13.000000,1.391187
6,Back Bay,Boston University,Wednesday,Night(0-6),14.382066,1.485906,13.579637,1.391433
...,...,...,...,...,...,...,...,...
497,West End,South Station,Friday,Night(0-6),15.728392,1.848727,14.619546,2.066955
498,West End,South Station,Monday,Night(0-6),15.723192,1.833155,14.435558,2.009084
501,West End,South Station,Thursday,Night(0-6),15.686901,1.835495,14.154671,2.008314
502,West End,South Station,Tuesday,Night(0-6),15.616097,1.866083,14.304613,1.998894


In [21]:

# ۱- ایجاد ستون‌های جدید برای میانگین قیمت و مسافت
# میانگین قیمت لیفت و اوبر (اگر یکی missing باشد، از دیگری استفاده شود)
comparison_df['avg_price'] = comparison_df[['avg_price_lyft', 'avg_price_uber']].mean(axis=1, skipna=True)

# میانگین مسافت لیفت و اوبر (اگر یکی missing باشد، از دیگری استفاده شود)
comparison_df['avg_distance'] = comparison_df[['avg_distance_lyft', 'avg_distance_uber']].mean(axis=1, skipna=True)

# ۲- حذف ستون‌های قیمت و مسافت قبلی
comparison_df = comparison_df.drop(
    columns=['avg_price_lyft', 'avg_price_uber', 'avg_distance_lyft', 'avg_distance_uber']
)


In [22]:
comparison_df

,source,destination,dayofweek,time_period,avg_price,avg_distance
0,Back Bay,Boston University,Friday,Night(0-6),14.250054,1.438286
1,Back Bay,Boston University,Monday,Night(0-6),14.012772,1.437433
4,Back Bay,Boston University,Thursday,Night(0-6),14.019937,1.444441
5,Back Bay,Boston University,Tuesday,Night(0-6),14.120226,1.439335
6,Back Bay,Boston University,Wednesday,Night(0-6),13.980852,1.438670
...,...,...,...,...,...,...
497,West End,South Station,Friday,Night(0-6),15.173969,1.957841
498,West End,South Station,Monday,Night(0-6),15.079375,1.921119
501,West End,South Station,Thursday,Night(0-6),14.920786,1.921904
502,West End,South Station,Tuesday,Night(0-6),14.960355,1.932488


In [23]:

# ۳- حذف داده‌های مربوط به شنبه و یکشنبه
comparison_df = comparison_df[~comparison_df['dayofweek'].isin(['Saturday', 'Sunday'])]


In [24]:
comparison_df

,source,destination,dayofweek,time_period,avg_price,avg_distance
0,Back Bay,Boston University,Friday,Night(0-6),14.250054,1.438286
1,Back Bay,Boston University,Monday,Night(0-6),14.012772,1.437433
4,Back Bay,Boston University,Thursday,Night(0-6),14.019937,1.444441
5,Back Bay,Boston University,Tuesday,Night(0-6),14.120226,1.439335
6,Back Bay,Boston University,Wednesday,Night(0-6),13.980852,1.438670
...,...,...,...,...,...,...
497,West End,South Station,Friday,Night(0-6),15.173969,1.957841
498,West End,South Station,Monday,Night(0-6),15.079375,1.921119
501,West End,South Station,Thursday,Night(0-6),14.920786,1.921904
502,West End,South Station,Tuesday,Night(0-6),14.960355,1.932488


In [27]:
import pandas as pd
import pulp

# خواندن داده‌ها
df = pd.read_csv("comparison_df.csv")

# تعریف ضریب هزینه برای مسافت (می‌توانید تغییر دهید)
cost_per_distance = 1.0

# محاسبه سود برای هر رکورد
df['profit'] = df['avg_price'] - cost_per_distance * df['avg_distance']

# ایجاد یک شناسه یکتا برای هر مسیر-روز (چون در داده‌ها time_period ثابت است، آن را حذف می‌کنیم)
df['route_day'] = df['source'] + '->' + df['destination'] + '_' + df['dayofweek']

# ایجاد مدل
model = pulp.LpProblem("Maximize_Profit", pulp.LpMaximize)

# متغیرهای تصمیم
x = pulp.LpVariable.dicts('x', df['route_day'], lowBound=0, upBound=1, cat='Binary')

# تابع هدف
model += pulp.lpSum([df.loc[i, 'profit'] * x[df.loc[i, 'route_day']] for i in df.index])

# محدودیت: هر روز بیش از 20 سفر نباشد
# ابتدا روزهای موجود را استخراج کن
days = df['dayofweek'].unique()

for day in days:
    # رکوردهای مربوط به این روز
    day_routes = df[df['dayofweek'] == day]['route_day']
    model += pulp.lpSum([x[route] for route in day_routes]) <= 20

# حل مدل
model.solve(pulp.PULP_CBC_CMD(msg=False))

# نمایش نتایج
print("Status:", pulp.LpStatus[model.status])
print("Total Profit =", pulp.value(model.objective))

# جدول نتایج
selected = []
for i in df.index:
    var = x[df.loc[i, 'route_day']]
    if var.varValue == 1:
        selected.append({
            'Route': df.loc[i, 'source'] + '->' + df.loc[i, 'destination'],
            'Day': df.loc[i, 'dayofweek'],
            'Price': df.loc[i, 'avg_price'],
            'Distance': df.loc[i, 'avg_distance'],
            'Profit': df.loc[i, 'profit']
        })

result_df = pd.DataFrame(selected)
print(f"\nSelected {len(result_df)} routes:\n")
print(result_df.to_string(index=False))

# تعداد سفرهای هر روز
print("\nTrips per day:")
print(result_df['Day'].value_counts().sort_index())

Status: Optimal
Total Profit = 1714.6076643584997

Selected 100 routes:

                                      Route       Day     Price  Distance    Profit
                 Back Bay->Haymarket Square    Monday 18.119265  2.372767 15.746498
                 Back Bay->Haymarket Square  Thursday 18.276959  2.396331 15.880628
                 Back Bay->Haymarket Square Wednesday 18.199800  2.363659 15.836141
                        Back Bay->North End    Friday 19.032472  2.885817 16.146654
                        Back Bay->North End    Monday 19.284519  2.828551 16.455969
                        Back Bay->North End  Thursday 19.357560  2.877959 16.479601
                        Back Bay->North End   Tuesday 19.751522  2.860098 16.891425
                        Back Bay->North End Wednesday 19.627404  2.882421 16.744983
      Boston University->Financial District    Friday 24.367343  4.543581 19.823762
      Boston University->Financial District    Monday 24.189949  4.535864 19.654085
   

In [26]:
import pulp
import pandas as pd
from collections import defaultdict

# آماده‌سازی داده‌ها
comparison_df = comparison_df.reset_index(drop=True)

# ایجاد نقشه‌ای از روزها به اندیس‌ها (برای کارایی بهتر)
day_to_indices = defaultdict(list)
for idx, row in comparison_df.iterrows():
    day_to_indices[row['dayofweek']].append(idx)

# ایجاد مدل
model = pulp.LpProblem("Route_Optimization", pulp.LpMaximize)

# متغیرهای تصمیم
x = pulp.LpVariable.dicts(
    'x', 
    range(len(comparison_df)), 
    lowBound=0, 
    upBound=1, 
    cat='Binary'
)

# تابع هدف (با استفاده از لیست‌های از پیش محاسبه شده برای کارایی بهتر)
prices = comparison_df['avg_price'].values
model += pulp.lpSum(prices[i] * x[i] for i in range(len(comparison_df)))

# محدودیت‌ها برای هر روز
for day, indices in day_to_indices.items():
    distances = comparison_df.loc[indices, 'avg_distance'].values
    model += pulp.lpSum(distances[j] * x[indices[j]] for j in range(len(indices))) <= 20, f"Day_{day}"

# حل با CBC (پیش‌فرض)
print("Solving the model...")
model.solve(pulp.PULP_CBC_CMD(msg=True, timeLimit=60, gapRel=0.05))

# تحلیل نتایج
print("\n" + "="*50)
print("SOLUTION ANALYSIS")
print("="*50)

if model.status == pulp.LpStatusOptimal:
    # شمارش مسیرهای انتخاب شده
    selected_count = sum(1 for i in range(len(comparison_df)) if pulp.value(x[i]) > 0.5)
    print(f"\n✓ Optimal solution found!")
    print(f"✓ Objective value: {pulp.value(model.objective):.2f}")
    print(f"✓ Number of selected routes: {selected_count}")
    
    # تحلیل تفصیلی بر اساس روز
    print("\n" + "-"*50)
    print("DAY-WISE ANALYSIS:")
    print("-"*50)
    
    summary_data = []
    for day in sorted(day_to_indices.keys()):
        indices = day_to_indices[day]
        day_distance = 0
        day_price = 0
        day_count = 0
        
        for idx in indices:
            if pulp.value(x[idx]) > 0.5:
                day_distance += comparison_df.loc[idx, 'avg_distance']
                day_price += comparison_df.loc[idx, 'avg_price']
                day_count += 1
        
        summary_data.append({
            'Day': day,
            'Selected Routes': day_count,
            'Total Distance': f"{day_distance:.2f}/20",
            'Total Price': f"{day_price:.2f}",
            'Utilization': f"{(day_distance/20)*100:.1f}%"
        })
    
    # نمایش جدول خلاصه
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    # ایجاد DataFrame نهایی
    selected_indices = [i for i in range(len(comparison_df)) if pulp.value(x[i]) > 0.5]
    final_selection = comparison_df.loc[selected_indices].copy()
    final_selection['is_selected'] = 1
    
    print(f"\n✓ Total price of selected routes: {final_selection['avg_price'].sum():.2f}")
    print(f"✓ Average price per selected route: {final_selection['avg_price'].mean():.2f}")
    
else:
    print(f"\n✗ Solution status: {pulp.LpStatus[model.status]}")
    
    # اگر جواب غیربهینه بود، سعی کن بهترین جواب را بگیر
    if hasattr(model, 'bestBound'):
        print(f"Best bound: {model.bestBound:.2f}")
    if hasattr(model, 'bestObjective'):
        print(f"Best objective: {model.bestObjective:.2f}")

ModuleNotFoundError: No module named 'pulp'

In [ ]:

# حل
solver = pyo.SolverFactory('glpk')
result = solver.solve(model)
